## Notebook Overview
## Single‑layer Exact Gaussian Process (GP) regression model

- This notebook contains a **single‑layer Exact Gaussian Process (GP) regression model** applied to radiomic features for ISUP grade prediction.  
- It performs a **full hyperparameter grid search** over:
  - PCA dimensions  
  - Kernel types  
  - Learning rates  
  - Noise initializations  
- The grid search trains a total of **336 Gaussian Process models**.  
- Each GP model is trained for **500 optimization iterations**.  
- **GPU acceleration** is used to speed up training for each individual GP model.  
- **Parallel execution across multiple GP models is not possible on a single GPU**, because:
  - Exact GPs require large covariance matrices and Cholesky decompositions.  
  - These operations fully occupy GPU memory and compute resources.  
  - Running multiple GPs simultaneously would cause memory conflicts and numerical instability.  
- For this reason, GP models are trained **sequentially on the GPU**, ensuring stable and reliable execution.
- Parts of the code were adapted and modified from the GPyTorch documentation: https://docs.gpytorch.ai/en/stable/examples/01_Exact_GPs/Simple_GP_Regression.html

In [1]:
import torch
import numpy as np
import gpytorch
import pandas as pd
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from deep_gp.gptorch_example import ExactGPModel
from deep_gp.preprocessing_data import load_data, undersample_class0, apply_smote
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
data = load_data()
df_new = undersample_class0(data)
df_resampled = apply_smote(df_new)

X_balanced = df_resampled.drop(columns=["case_ISUP"])
y_balanced = df_resampled["case_ISUP"]


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42
)

# Scale features
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

# Scale target
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1,1)).ravel()
y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1,1)).ravel()


In [ ]:
results = []

pca_components_list = [5, 10, 15, 20, 30, 40, 50]
kernels = ["matern_15", "matern_25", "rbf_ard"]
learning_rates = [0.05, 0.01, 0.001]
noise_values = ["learned", 0.01, 0.05, 0.1, 0.2]

for n_comp in pca_components_list:
    print(f"\n PCA = {n_comp} components")

    # PCA transform
    pca = PCA(n_components=n_comp, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca  = pca.transform(X_test_scaled)

    # Move tensors to GPU
    X_train_tensor = torch.tensor(X_train_pca, dtype=torch.float32).to(device)
    X_test_tensor  = torch.tensor(X_test_pca,  dtype=torch.float32).to(device)

    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).to(device)
    y_test_tensor  = torch.tensor(y_test_scaled,  dtype=torch.float32).to(device)

    for kernel in kernels:
        for lr in learning_rates:
            for noise_init in noise_values:

                print(f"\n--- Kernel: {kernel} | LR: {lr} | Noise: {noise_init} ---")

                torch.manual_seed(42)

                likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)

                if noise_init == "learned":
                    print("   Noise: learned (GPyTorch default, trainable)")
                else:
                    likelihood.noise = noise_init
                    print(f"   Noise init: {noise_init} (trainable)")

                model = ExactGPModel(
                    X_train_tensor, y_train_tensor, likelihood, kernel_type=kernel
                ).to(device)

                model.train()
                likelihood.train()

                optimizer = torch.optim.Adam(model.parameters(), lr=lr)
                mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

                # tqdm progress bar for 500 iterations
                for i in tqdm(range(500), desc="Training GP", leave=False):
                    optimizer.zero_grad()
                    output = model(X_train_tensor)
                    loss = -mll(output, y_train_tensor)
                    loss.backward()
                    optimizer.step()

                # Predict
                model.eval()
                likelihood.eval()

                with torch.no_grad(), gpytorch.settings.fast_pred_var():
                    preds = model(X_test_tensor)

                # Move predictions back to CPU
                mean_scaled = preds.mean.cpu().numpy()
                var_scaled  = preds.variance.cpu().numpy()
                std_scaled  = np.sqrt(var_scaled)

                # Back to ISUP scale
                mean = scaler_y.inverse_transform(mean_scaled.reshape(-1,1)).ravel()
                std  = std_scaled * scaler_y.scale_[0]

                y_test_original = y_test.values

                mse = mean_squared_error(y_test_original, mean)
                r2  = r2_score(y_test_original, mean)
                mean_unc = std.mean()

                print(f"MSE: {mse:.4f} | R²: {r2:.4f} | Mean Uncertainty: {mean_unc:.4f}")

                # Build uncertainty dataframe (for best-model analysis)
                df_unc = pd.DataFrame({
                    "true": y_test_original,
                    "pred": mean,
                    "std": std
                })

                results.append([
                    n_comp, kernel, lr, noise_init, mse, r2, mean_unc, df_unc
                ])



 PCA = 5 components

--- Kernel: matern_15 | LR: 0.05 | Noise: learned ---
   Noise: learned (GPyTorch default, trainable)


MSE: 2.2500 | R²: 0.3176 | Mean Uncertainty: 1.5893

--- Kernel: matern_15 | LR: 0.05 | Noise: 0.01 ---
   Noise init: 0.01 (trainable)


MSE: 2.2519 | R²: 0.3171 | Mean Uncertainty: 1.5909

--- Kernel: matern_15 | LR: 0.05 | Noise: 0.05 ---
   Noise init: 0.05 (trainable)


MSE: 2.2503 | R²: 0.3175 | Mean Uncertainty: 1.5891

--- Kernel: matern_15 | LR: 0.05 | Noise: 0.1 ---
   Noise init: 0.1 (trainable)


MSE: 2.2504 | R²: 0.3175 | Mean Uncertainty: 1.5866

--- Kernel: matern_15 | LR: 0.05 | Noise: 0.2 ---
   Noise init: 0.2 (trainable)


...

MSE: 2.3277 | R²: 0.2941 | Mean Uncertainty: 1.2592

--- Kernel: rbf_ard | LR: 0.001 | Noise: 0.01 ---
   Noise init: 0.01 (trainable)


MSE: 2.0404 | R²: 0.3812 | Mean Uncertainty: 1.5016

--- Kernel: rbf_ard | LR: 0.001 | Noise: 0.05 ---
   Noise init: 0.05 (trainable)


MSE: 2.0497 | R²: 0.3784 | Mean Uncertainty: 1.4984

--- Kernel: rbf_ard | LR: 0.001 | Noise: 0.1 ---
   Noise init: 0.1 (trainable)


MSE: 2.0615 | R²: 0.3748 | Mean Uncertainty: 1.4837

--- Kernel: rbf_ard | LR: 0.001 | Noise: 0.2 ---
   Noise init: 0.2 (trainable)


MSE: 2.0917 | R²: 0.3656 | Mean Uncertainty: 1.4469


In [6]:
df_results = pd.DataFrame(
    results,
    columns=["PCA", "Kernel", "LR", "Noise", "MSE", "R2", "Mean_Uncertainty", "UncertaintyDF"]
)


In [7]:
best = df_results.sort_values(["R2", "Mean_Uncertainty"], ascending=[False, True]).iloc[0]

print("\n===== BEST MODEL FOUND =====")
print(f"PCA components:   {best['PCA']}")
print(f"Kernel:           {best['Kernel']}")
print(f"Learning rate:    {best['LR']}")
print(f"Noise setting:    {best['Noise']}")
print(f"MSE:              {best['MSE']:.4f}")
print(f"R²:               {best['R2']:.4f}")
print(f"Mean uncertainty: {best['Mean_Uncertainty']:.4f}")



===== BEST MODEL FOUND =====
PCA components:   40
Kernel:           matern_25
Learning rate:    0.05
Noise setting:    0.1
MSE:              1.0163
R²:               0.6918
Mean uncertainty: 1.2776


In [9]:
df_unc_best = best["UncertaintyDF"]

unc_by_class = df_unc_best.groupby("true")["std"].mean()

print("\n===== MEAN UNCERTAINTY (STD) PER ISUP CLASS =====")
for cls, std in unc_by_class.items():
    print(f"ISUP {cls}:  std = {std:.4f}")



===== MEAN UNCERTAINTY (STD) PER ISUP CLASS =====
ISUP 0:  std = 1.4362
ISUP 1:  std = 1.3872
ISUP 2:  std = 1.4074
ISUP 3:  std = 1.2334
ISUP 4:  std = 0.9226
ISUP 5:  std = 1.1138


In [10]:
num_models = (
    len(pca_components_list)
    * len(kernels)
    * len(learning_rates)
    * len(noise_values)
)

print(f"Total models to be tested: {num_models}")


Total models to be tested: 315
